In [2]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [3]:
"""
Load data from files and create the following tables.
    facilities -> facilities.csv
    members -> members.csv
    bookings -> bookings.csv
Choose appropriate data types to best represent the data fields.
"""
# defining schemas

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType # type: ignore

facilities_schema = StructType([
    StructField("facid", IntegerType(), nullable=False), # though we have mentioned nullable False here the schema does not show that because in spark constraints during read are not strict constraints, they are parsing instructions
    StructField("fac_name", StringType()),
    StructField("membercost", IntegerType()),
    StructField("guestcost", IntegerType()),
    StructField("initialoutlay", IntegerType()),
    StructField("monthlymaintenance", IntegerType())
])

members_schema = StructType([
    StructField("memid", IntegerType(), nullable=False), # though we have mentioned nullable False here the schema does not show that because in spark constraints during read are not strict constraints, they are parsing instructions
    StructField("surname", StringType()),
    StructField("firstname", StringType()),
    StructField("address", StringType()),
    StructField("zipcode", StringType()),
    StructField("telephone", StringType()),
    StructField("recommendedby", IntegerType()),
    StructField("joindate", TimestampType())
])

bookings_schema = StructType([
    StructField("bookid", IntegerType(), nullable=False), # though we have mentioned nullable False here the schema does not show that because in spark constraints during read are not strict constraints, they are parsing instructions
    StructField("facid", IntegerType()),
    StructField("memid", IntegerType()),
    StructField("starttime", TimestampType()),
    StructField("slots", IntegerType())
])

In [4]:
# loading into dataframe

facilities_df = spark.read.format("csv")\
                        .option("header", True)\
                        .schema(facilities_schema)\
                        .load(path = "/home/jovyan/work/data/facilities.csv")
facilities_df.show()
facilities_df.printSchema()

+-----+---------------+----------+---------+-------------+------------------+
|facid|       fac_name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+---------------+----------+---------+-------------+------------------+
|    0| Tennis Court 1|         5|       25|        10000|               200|
|    1| Tennis Court 2|         5|       25|         8000|               200|
|    2|Badminton Court|         0|     NULL|         4000|                50|
|    3|   Table Tennis|         0|        5|          320|                10|
|    4| Massage Room 1|        35|       80|         4000|              3000|
|    5| Massage Room 2|        35|       80|         4000|              3000|
|    6|   Squash Court|      NULL|     NULL|         5000|                80|
|    7|  Snooker Table|         0|        5|          450|                15|
|    8|     Pool Table|         0|        5|          400|                15|
+-----+---------------+----------+---------+-------------+------

In [5]:
members_df = spark.read.format("csv")\
                    .option("header", True)\
                    .schema(members_schema)\
                    .load(path = "/home/jovyan/work/data/members.csv")
members_df.show()
members_df.printSchema()

+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|memid|  surname|firstname|             address|zipcode|     telephone|recommendedby|           joindate|
+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|    0|    GUEST|    GUEST|               GUEST|      0|(000) 000-0000|         NULL|2022-07-01 00:00:00|
|    1|    Smith|   Darren|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2022-07-02 12:02:05|
|    2|    Smith|    Tracy|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2022-07-02 12:08:23|
|    3|   Rownam|      Tim|23 Highway Way, B...|  23423|(844) 693-0723|         NULL|2022-07-03 09:32:15|
|    4| Joplette|   Janice|20 Crossing Road,...|    234|(833) 942-4710|            1|2022-07-03 10:25:05|
|    5|  Butters|   Gerald|1065 Huntingdon A...|  56754|(844) 078-4130|            1|2022-07-09 10:44:09|
|    6|    Tracy|   Burton|3 Tunisia Drive, ..

In [6]:
bookings_df = spark.read.format("csv")\
                    .option("header", True)\
                    .schema(bookings_schema)\
                    .load(path = "/home/jovyan/work/data/bookings.csv")
bookings_df.show()
bookings_df.printSchema()

+------+-----+-----+-------------------+-----+
|bookid|facid|memid|          starttime|slots|
+------+-----+-----+-------------------+-----+
|     0|    3|    1|2022-07-03 11:00:00|    2|
|     1|    4|    1|2022-07-03 08:00:00|    2|
|     2|    6|    0|2022-07-03 18:00:00|    2|
|     3|    7|    1|2022-07-03 19:00:00|    2|
|     4|    8|    1|2022-07-03 10:00:00|    1|
|     5|    8|    1|2022-07-03 15:00:00|    1|
|     6|    0|    2|2022-07-04 09:00:00|    3|
|     7|    0|    2|2022-07-04 15:00:00|    3|
|     8|    4|    3|2022-07-04 13:30:00|    2|
|     9|    4|    0|2022-07-04 15:00:00|    2|
|    10|    4|    0|2022-07-04 17:30:00|    2|
|    11|    6|    0|2022-07-04 12:30:00|    2|
|    12|    6|    0|2022-07-04 14:00:00|    2|
|    13|    6|    1|2022-07-04 15:30:00|    2|
|    14|    7|    2|2022-07-04 14:00:00|    2|
|    15|    8|    2|2022-07-04 12:00:00|    1|
|    16|    8|    3|2022-07-04 18:00:00|    1|
|    17|    1|    0|2022-07-05 17:30:00|    3|
|    18|    2

In [ ]:
# loading dfs into table

facilities_df.write.mode("overwrite").saveAsTable("spark_db.facilities")
members_df.write.mode("overwrite").saveAsTable("spark_db.members")
bookings_df.write.mode("overwrite").saveAsTable("spark_db.bookings")

In [ ]:
spark.sql("SELECT * FROM spark_db.facilities").show()
spark.sql("SELECT * FROM spark_db.members").show()
spark.sql("SELECT * FROM spark_db.bookings").show()

+-----+---------------+----------+---------+-------------+------------------+
|facid|       fac_name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+---------------+----------+---------+-------------+------------------+
|    0| Tennis Court 1|         5|       25|        10000|               200|
|    1| Tennis Court 2|         5|       25|         8000|               200|
|    2|Badminton Court|         0|     NULL|         4000|                50|
|    3|   Table Tennis|         0|        5|          320|                10|
|    4| Massage Room 1|        35|       80|         4000|              3000|
|    5| Massage Room 2|        35|       80|         4000|              3000|
|    6|   Squash Court|      NULL|     NULL|         5000|                80|
|    7|  Snooker Table|         0|        5|          450|                15|
|    8|     Pool Table|         0|        5|          400|                15|
+-----+---------------+----------+---------+-------------+------